# Day 5 - Scikit-learn Pipelines & Tuned Mini-Project
By the end of this notebook, we will be able to:
- Explain the importance of Scikit-learn Pipelines.
- Explain how Pipelines can help avoid data leakage.
- Incorporate the engineered features from Day 4.
- Preprocess numerical and categorical features separately using ColumnTransformer.
- Construct a complete preprocessing + modeling Pipeline.
- Tune the entire Pipeline using GridSearchCV.
- Use 5-fold cross-validation.
- Evaluate our tuned Pipeline on a held-out test set.
- Compare our final model to a baseline.

## Import Libraries

In [98]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,classification_report)

## Dataset Used
We will use the same Student Performance  dataset,this data includes information about students' academic, personal and environmental backgrounds.
The original target variable is **final_exam_score**,in today's classification setting, we are going to construct:
- Pass = 1 if Exam_Score >= 60
- Pass = 0 if Exam_Score < 60

## Load the Dataset

In [99]:
df=pd.read_csv("student_performance_dataset.csv")

In [100]:
df.head(10)

,student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,1,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9,100.0,A
1,2,Female,6.3,100.0,5.7,High School,Yes,Yes,Yes,75.5,100.0,A
2,3,Male,4.9,85.3,7.9,Bachelors,Yes,No,Yes,88.5,97.3,A
3,4,Male,2.6,77.5,8.0,NaN,Yes,Yes,No,85.1,83.8,B
4,5,Male,2.2,89.6,4.6,Bachelors,Yes,No,Yes,61.8,68.3,D
5,6,Female,4.2,78.2,5.7,High School,Yes,No,No,79.8,81.6,B
6,7,Male,1.5,100.0,5.0,High School,Yes,Yes,Yes,58.6,69.6,D
7,8,Male,6.2,86.4,6.0,High School,Yes,Yes,No,78.4,88.0,B
8,9,Male,5.3,81.3,6.7,Bachelors,Yes,No,No,63.8,84.4,B
9,10,Female,2.8,86.8,5.1,Bachelors,Yes,Yes,No,57.7,74.3,C


We load the dataset into pandas DataFrame ,the first rows help us understand the structure of the data.

### Dataset Overview

In [101]:
df.shape

(1000, 12)

In [102]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   student_id                  1000 non-null   int64  
 1   gender                      1000 non-null   str    
 2   study_time_hours            1000 non-null   float64
 3   attendance_percent          1000 non-null   float64
 4   sleep_hours                 1000 non-null   float64
 5   parental_education          898 non-null    str    
 6   internet_access             1000 non-null   str    
 7   extracurricular_activities  1000 non-null   str    
 8   part_time_job               1000 non-null   str    
 9   previous_grade              1000 non-null   float64
 10  final_exam_score            1000 non-null   float64
 11  final_grade                 1000 non-null   str    
dtypes: float64(5), int64(1), str(6)
memory usage: 93.9 KB


There are both numeric and categorical variables in the dataset.
It is crucial for today’s tasks because numeric and categorical variables require different preprocessing steps.

In [103]:
df.isnull().sum()

student_id                      0
gender                          0
study_time_hours                0
attendance_percent              0
sleep_hours                     0
parental_education            102
internet_access                 0
extracurricular_activities      0
part_time_job                   0
previous_grade                  0
final_exam_score                0
final_grade                     0
dtype: int64

### Missing Values
There are some missing values in column  **parental_education**.
Instead of filling these values manually before splitting the data,we will handle them inside the Pipeline.It is essential to do so since the imputation technique will be fit only on the right training data during cross-validation.

## Define the target 

In [104]:
df["Pass"] = (df["final_exam_score"] >= 60).astype(int)
df["Pass"].value_counts()

Pass
1    988
0     12
Name: count, dtype: int64

### Class Imbalance
Class imbalance is another issue with the target feature, where the vast majority of students fall into the category of Pass, and few students belong to the Fail category.
As a result of such an imbalance, the F1 score for a majority baseline may be extremely high.
Thus, a high F1 score does not always indicate model performance since a model may easily classify students as passes.That is why we will pay special attention to the Fail class and evaluate its F1 score separately.

### Why do we need to delete final grade?
final_grade is the exact variable used to determine Pass.
When final_grade remains in X, we will be providing the model with the answer.
This will lead to an obvious case of data leakage.
So:
final_exam_Score --> used for target creation
final_grade --> not used as a feature
### Why do we remove student_id?
student_id is only an identifier used to distinguish students. It does not represent meaningful information about their academic performance. Keeping it could allow the model to learn accidental patterns from the ID numbers, so it is removed before training.

## Feature Engineering from Day 4
The same engineered features will be used as were used on Day 4.
### Feature 1 – Study_Attendance_Index
This is an attendance value as a fraction rather than a percentage.
### Feature 2 – Past_Score_Study_Ratio
This is a value representing the previous scores as a fraction.
### Feature 3 – Study_level
This is a classification based on the number of study hours.
These are not new features but are engineered features from Day 4.
They are the engineered features from Day 4, now being incorporated into
a complete Pipeline.

In [105]:
def add_day4_features(X):
    x = X.copy()
    
    # Day 4 Feature 1
    x["Study_Attendance_Index"] = (x["study_time_hours"] * x["attendance_percent"] / 100)
    # Day 4 Feature 2
    x["Past_Score_Study_Ratio"] = x["previous_grade"] / (x["study_time_hours"] + 1)
    # Day 4 Feature 3
    x["study_level"] = pd.cut(
        x["study_time_hours"],
        bins=[-np.inf, 2, 4, 8, np.inf],
        labels=["Very Low", "Low", "Medium", "High"]
    )
    return x

## Define X and y
We remove:
- final_exam_score because it was used to create the target.
- Pass because it is the target itself.
All remaining columns considered input features.

In [106]:
X = df.drop(columns=["final_exam_score","final_grade","student_id", "Pass"])
y = df["Pass"]
print("x shape is :", X.shape)
print("y shape is:", y.shape)

x shape is : (1000, 9)
y shape is: (1000,)


# 5. Train-Test Split
Data splitting is done before applying any preprocessing technique,the test data will be left untouched until the final evaluation.
Stratification is applied to ensure that the ratio of Pass/Fail students is approximately preserved in both test and training data.

In [107]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (800, 9)
Test set: (200, 9)


## Why should we split before doing preprocessing?
Scaling and imputing techniques in the preprocessing stage learn something from the data.
For instance, the mean and standard deviation are learned by StandardScaler.
When these numbers are computed using the full data before splitting,the test set will affect the preprocessing step,this will lead to data leakage.Thus, the test set should be kept unseen until evaluation time.

## Determine Numerical and Categorical Features
We now have to determine which features are numerical and which are categorical.
Numerical features will include:
- median imputation
- standard scaling
Categorical features will include:
- mode imputation
- one hot encoding

In [108]:
numeric_cols = ["study_time_hours","attendance_percent","sleep_hours","previous_grade","Study_Attendance_Index","Past_Score_Study_Ratio"]
categorical_cols = ["gender","parental_education","internet_access","extracurricular_activities","part_time_job","study_level"]
print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

Numeric columns: ['study_time_hours', 'attendance_percent', 'sleep_hours', 'previous_grade', 'Study_Attendance_Index', 'Past_Score_Study_Ratio']
Categorical columns: ['gender', 'parental_education', 'internet_access', 'extracurricular_activities', 'part_time_job', 'study_level']


## Numeric Preprocessing
For numeric features, we create a simple Pipeline:
- SimpleImputer
- StandardScaler
SimpleImputer is used for missing numeric values,StandardScaler is used for numeric features.

In [109]:
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler())])

## Categorical Preprocessing
The preprocessing techniques for categorical features are:
- SimpleImputer
- OneHotEncoder
"handle_unknown="ignore" enables the Pipeline to handle categories that could be present in the testing set, but not the training set.

In [110]:
categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),("encoder", OneHotEncoder(handle_unknown="ignore"))])

# 10. Construct the Full Pipeline
We now merge all the main components into one Pipeline.
The order is:
1. Feature engineering from Day 4
2. ColumnTransformer pre-processing
3. Random Forest classifier
It means that the whole pipeline will be contained in ONE object.More importantly, when GridSearchCV does cross-validation,it is done properly in every fold.

In [111]:
feature_engineering = FunctionTransformer(add_day4_features)
preprocessor = ColumnTransformer(
    transformers=[("num", numeric_pipeline, numeric_cols),("cat", categorical_pipeline, categorical_cols)])
pip = Pipeline([("features", feature_engineering),("pre", preprocessor),("model", RandomForestClassifier(random_state=42,class_weight="balanced"))])

ColumnTransformer allows us to apply the numerical and categorical preprocessing separately.

## What is so good about using this pipeline versus just executing each step?
Without using a Pipeline, there may be mistakes like:
- fitting the scaler to the entire data set
- encoding the validation data incorrectly
- doing inconsistent feature engineering
- leaking information through cross-validation
Using the Pipeline allows you to connect the steps and have Scikit-learnapply them correctly.

## Baseline Model
A majority-class baseline is used as a simple reference point. Since the target is highly imbalanced, we evaluate the baseline using F1 for the Fail class.

In [112]:
majority_class = y_train.mode()[0]
baseline_pred = np.full(len(y_test),majority_class)
baseline_f1_fail = f1_score(y_test,baseline_pred,pos_label=0,zero_division=0)
print("Majority class:", majority_class)
print("Baseline Fail F1:", baseline_f1_fail)

Majority class: 1
Baseline Fail F1: 0.0


## Hyperparameter Grid
In this section, we will optimize the Random Forest classifier.
The hyperparameters which will be optimized are:
- n_estimators: number of trees
- max_depth: depth of the trees
- min_samples_split: minimum samples required for a split in a node
As our model is wrapped inside a Pipeline object, we need to call the hyperparameter name as follows:
model__hyperparameter_name

In [113]:
param_grid = {"model__n_estimators": [100, 200],"model__max_depth": [5, 10, None],"model__min_samples_split": [2, 5]}

## GridSearchCV
GridSearchCV tunes the complete Pipeline using 5-fold cross-validation,the F1 score for the Fail class is used as the scoring metric because the target is highly imbalanced and identifying students at risk of failing is important.

In [114]:
fail_f1_scorer = make_scorer(f1_score,pos_label=0)

In [115]:
grid = GridSearchCV(estimator=pip,param_grid=param_grid,cv=5,scoring=fail_f1_scorer,n_jobs=-1)
grid.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [5, 10, ...], 'model__min_samples_split': [2, 5], 'model__n_estimators': [100, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","make_scorer(f..., pos_label=0)"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",Tru

## Fit the Whole Pipeline
Let us now fit the GridSearchCV with the training set only.
GridSearchCV will:
- Perform Day 4 feature engineering.
- Perform ColumnTransformer preprocessing.
- Train the Random Forest.
- Perform evaluation using 5-fold cross validation.
- Do this for all combinations of hyperparameters.
- Choose the best combination.

## Best Hyperparameters
best_params_ indicates the set of hyperparameters which provided the 
highest mean F1 score.
best_score_ is the mean F1 score obtained using cross-validation.

In [116]:
print("Best Parameters:")
print(grid.best_params_)
print("Best CV F1 Score:")
print(grid.best_score_)

Best Parameters:
{'model__max_depth': 5, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Best CV F1 Score:
0.12380952380952381


# 16. Best Pipeline
best_estimator_ is the entire Pipeline with the optimal hyperparameters selected by GridSearchCV.
It includes:
- Feature engineering for Day 4
- Numerical preprocessing
- Categorical preprocessing
- Random Forest
- best hyperparameters

In [117]:
best_pipeline = grid.best_estimator_
best_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('features', ...), ('pre', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['gender','study_time_hours','attendance_percent',..., 'extracurricular_activities','part_time_job','previous_grade']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function add...002021A8E0220>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False


# 17. Final Test Evaluation
The test set was not used in GridSearchCV,now we use it for the final evaluation.
This will give us an estimation of the performance of the tuned Pipeline on completely unseen data.

In [118]:
y_pred = best_pipeline.predict(X_test)

In [119]:
accuracy = accuracy_score(y_test, y_pred)
precision_fail = precision_score(y_test,y_pred,pos_label=0)
recall_fail = recall_score(y_test,y_pred,pos_label=0)
f1_fail = f1_score(y_test,y_pred,pos_label=0)
print(f"Accuracy      : {accuracy}")
print(f"Precision Fail: {precision_fail}")
print(f"Recall Fail   : {recall_fail}")
print(f"F1 Score Fail : {f1_fail}")

Accuracy      : 0.965
Precision Fail: 0.0
Recall Fail   : 0.0
F1 Score Fail : 0.0


In [120]:
print(classification_report(
    y_test,
    y_pred,
    target_names=["Fail", "Pass"],
    zero_division=0
))

              precision    recall  f1-score   support

        Fail       0.00      0.00      0.00         2
        Pass       0.99      0.97      0.98       198

    accuracy                           0.96       200
   macro avg       0.49      0.49      0.49       200
weighted avg       0.98      0.96      0.97       200



## Compare Against the Baseline
We now compare the tuned Pipeline with the simple majority-class baseline.
The important metric for this project is F1 score, the tuned Pipeline has a higher F1 score than the baseline then the model is learning useful patterns from the student features.

In [121]:
print(f"Baseline Fail F1:       {baseline_f1_fail}")
print(f"Tuned Pipeline Fail F1: {f1_fail}")
print(f"Improvement:             {f1_fail - baseline_f1_fail}")

Baseline Fail F1:       0.0
Tuned Pipeline Fail F1: 0.0
Improvement:             0.0


## Cross-validation score VS test score
The optimal value of cross-validation score and the test score are not necessarily the same.
CV score refers to the mean of the performances in the validation
folds employed in GridSearchCV.
Test score refers to the performance on an entirely unseen dataset.
A small difference between these two scores is expected.
The main thing to consider is that the test set was not involved in hyperparameter tuning.

In [122]:
print(f"Best CV Fail F1: {grid.best_score_}")
print(f"Test Fail F1:    {f1_fail}")

Best CV Fail F1: 0.12380952380952381
Test Fail F1:    0.0


## Final Evaluation
Compare:
- Baseline F1
- Best Cross-Validation F1
- Final Test F1


In [123]:
results = pd.DataFrame({"Metric": ["Baseline Fail F1","Best CV Fail F1","Final Test Fail F1"],"Score": [baseline_f1_fail,grid.best_score_,f1_fail]})
results

,Metric,Score
0,Baseline Fail F1,0.00000
1,Best CV Fail F1,0.12381
2,Final Test Fail F1,0.00000


### Analysis of the Final Results
The Pipeline, after tuning, is evaluated on the test set just once. The test set has not been seen during training, cross-validation, and hyperparameter tuning.
The dataset is highly imbalanced, with only a small number of Fail cases. The tuned Pipeline achieved a higher cross-validation Fail F1 than the majority baseline, but its Fail F1 on the held-out test set was 0. This shows that the model did not successfully identify the Fail cases in this particular test split.Therefore, the test result should be interpreted cautiously, especially because only two Fail cases were present in the test set.

# Hands-On Lab
## Step 1: Build a Pipeline with a ColumnTransformer handling numeric (scaling) and categorical (encoding) columns.

The Pipeline consists of three major components:
1. Feature engineering 
2. Preprocessing with ColumnTransformer
3. Random Forest model 
ColumnTransformer enables us to perform preprocessing for numerical and categorical data separately.
Preprocessing is included in the Pipeline in order to make it fitted correctly during cross-validation and without using any information from the validation splits.By including the Day 4 feature engineering step into the Pipeline we ensure the reproducibility of the whole process and include the feature engineering step into the end-to-end process.

## Step 2: Add the engineered features from Day 4 into the workflow.
Engineered Features for Day 4 need to be incorporated into the Day 5 workflow.
They include:
- Study_Attendance_Index
- Past_Score_Study_Ratio
- study_level
Engineered Features for Day 4 are used again in Day 5 since Day 5 is a continuation
of the workflow from Day 4.Day 5 is not aimed at creation of new features, but rather at incorporation of the engineered features into an actual Pipeline.It enables feature engineering, data preprocessing, modeling, and hyperparameters tuning to function as one combined workflow.


## Step 3: Tune the full pipeline with GridSearchCV and 5-fold cross-validation
GridSearchCV was used to tune the complete Pipeline using 5-fold cross-validation. The Fail-class F1 score was used as the scoring metric because the target is highly imbalanced.

## Step 4: Evaluate the final tuned pipeline once on the held-out test set and report the metric against a baseline.
The tuned Pipeline was evaluated once on the held-out test set and compared with the majority-class baseline. The cross-validation Fail F1 was 0.124, while the final test Fail F1 was 0.0. This result should be interpreted cautiously because the test set contained only two Fail cases.

## Final Summary
This project combined Day 4 feature engineering with a complete Scikit-learn Pipeline. The Pipeline performed feature engineering, numeric and categorical preprocessing, and Random Forest classification in one workflow. GridSearchCV with 5-fold cross-validation was used to tune the pipeline without data leakage. The final model was evaluated once on the held-out test set using Fail-class F1 and compared with a majority-class baseline.